# π0.5 軽量版（VLADrop Drop9）× LIBERO-Plus 追加学習（Colab）

**提出動画の後半で「π0.5」として見せているモデルを作った学習**を、Colab で再現する notebook。
起点は、π0.5 を LIBERO-Plus で学習し、言語モデルの 18 層のうち 9 層を落とした軽量版（VLADrop の Drop9）。
これを、公開されている VLADrop の学習コード（openpi ベース）で**全パラメータ**追加学習する。

```text
LeRobot データセット（HF） ─▶ この notebook
                              ├ VLADrop の学習環境を作る（fork の依存 + v3.0 を読むための書き換え）
                              ├ 追加学習（全パラメータ、train_pytorch.py）
                              └ checkpoint を HF へアップロード（任意）
```

**上から順に実行する。書き換えるのは「1. 設定」のセルだけ。**
既定では公開データ `lerobot/libero_plus` を丸ごと（約 16 GB）落として学習する。
teleop のデータセットができたら `DATASET_REPO` を差し替え、`DATASET_REVISION` を `None` にする。

**この notebook は学習と checkpoint の保存までで、シミュレータでの評価は入れていない**（下の「学習済みモデルについて」）。

## 動画のモデルを作ったときの条件（parc2026、2026-09-20）

| | |
| :--- | :--- |
| 初期重み | [`s1ghhh/VLADrop_pi05_LIBERO-Plus_Drop9Block_GateProbe`](https://huggingface.co/s1ghhh/VLADrop_pi05_LIBERO-Plus_Drop9Block_GateProbe) @ `5760f99f` |
| 学習コード | [`s1ghhh/VLADrop`](https://github.com/s1ghhh/VLADrop) @ `782d3e56` の `models/pi0.5/openpi/scripts/train_pytorch.py`（設定 `pi05_libero_plus`） |
| 教師データ | 元の LIBERO の課題を逆向きにした 8 課題の、人の teleop 91 本から作った 415 episode。91 本全体と、対象が 1 cm 持ち上がった時点で分けた前半・後半（各 81 本 × 指示文 2 通り）。30,599 frame、指示文 36 通り。非公開。LeRobot v2.1 に変換して読ませた |
| 画像 | 教師データは 128×128。学習では pad 付きで 224×224 に拡大して読む |
| 学習 | 全パラメータ、600 step、batch 16、warmup 100 step、lr 5e-5 → 5e-6（cosine）、AdamW（β 0.9 / 0.95、weight decay 1e-10）、勾配クリップ 1.0、gradient checkpointing。EMA 0.999 を渡したが、PyTorch の学習経路（`train_pytorch.py`）では適用されない |
| GPU | Colab A100 40 GB。1 step 約 1.6 秒（600 step で約 22 分）、VRAM のピーク 28.9 GB |
| 評価 | 逆向きの課題 8 種 × 指示文 2 通りの 16 問、各 1 試行。学習前 0/16 → 学習後はゴール到達 13/16、うち対象外の物体を動かさない（変位 1 mm 以下）成功 8/16 |

この notebook の既定値は、データ以外はこの条件と同じにしてある。

### データの読み方だけが違う

VLADrop の fork が固定している LeRobot は v2.1 形式しか読めない。動画のモデルでは教師データを v2.1 に変換したが、
この notebook では **fork の LeRobot を 0.4.4 に入れ替え、データを読む関数を少し書き換えて、v3.0 形式のまま読む**
（このリポジトリの `dataset/` が書き出す形式と、`lerobot/libero_plus` がどちらも v3.0 のため）。
parc2026 では同じ学習条件をこの読み方でも回し、loss の推移が v2.1 版と一致すること、
上の 16 問で、対象外の物体を動かさない成功が 7/16（v2.1 版は 8/16）になることを確かめている。学習ループやモデルの構造には手を入れていない。

## 用意するもの

| | |
| :--- | :--- |
| GPU | ランタイム → ランタイムのタイプを変更 → **A100**（VRAM 40 GB 以上）。全パラメータを学習するので L4 / T4 では足りない |
| Hugging Face token | 左の鍵アイコン（Secrets）に `HF_TOKEN` として登録する。非公開のデータセットを読む・アップロードするなら、その権限つき |

## データセットの条件

`lerobot/libero_plus` と同じ形式（LeRobot v3.0）にする。fork の設定 `pi05_libero_plus` がこのキー名で読む。

| key | 型 | 内容 |
| :--- | :--- | :--- |
| `observation.images.front` | video 256×256×3 | `agentview` カメラ。**env の生画像を 180° 回転**（上下・左右とも反転）したもの |
| `observation.images.wrist` | video 256×256×3 | `robot0_eye_in_hand` カメラ。同じく 180° 回転 |
| `observation.state` | float32 (8,) | eef 位置 3 + eef 姿勢（クォータニオン → 軸角）3 + グリッパ指の位置 2 |
| `action` | float32 (7,) | OSC_POSE の差分 6 + グリッパ 1（teleop の `actions` そのまま） |
| `task` | str | 指示文（teleop の録画 JSON の `language`） |

fps は 20（LIBERO の control_freq と同じ）。**データセットの全 episode を学習に使う。**

## 1. 設定

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------- データ
# LeRobot v3.0 形式のデータセット（全 episode を学習に使う）
# teleop のデータセットを HF に上げたら、その repo に差し替える（revision は None で最新）
DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = "f3f49f426d75030177b18778374005bc12ccd588"

# ---------------------------------------------------------------- 初期重み
# π0.5 を LIBERO-Plus で学習し、言語モデルの 18 層のうち 9 層を落とした軽量版
BASE_MODEL_REPO = "s1ghhh/VLADrop_pi05_LIBERO-Plus_Drop9Block_GateProbe"
BASE_MODEL_REVISION = "5760f99f42ece92c4b98c140685f5a090b0bb232"
# 落とした層の番号（attention と MLP で同じ）。重みに同梱の metadata.pt の値で、HF の README の記載とは食い違う
LLM_DROP_LIST = [7, 10, 11, 12, 13, 14, 15, 16, 17]

# ---------------------------------------------------------------- 学習（動画のモデルと同じ条件）
# 全パラメータを学習する（fork の既定）
STEPS = 600
BATCH_SIZE = 16
WARMUP_STEPS = 100
PEAK_LR = 5e-5
DECAY_LR = 5e-6
EMA_DECAY = 0.999         # fork に渡すが、PyTorch の学習経路では適用されない（動画のモデルも同じ）
GRADIENT_CHECKPOINTING = True
NUM_WORKERS = 4
SMOKE_STEPS = 20          # 本番の前に、この step 数で最後まで通るか（メモリに載るか）を確かめる。0 なら飛ばす
LOG_FREQ = 20             # loss を表示する間隔（step）
RUN_NAME = f"drop9_ft_s{STEPS}_b{BATCH_SIZE}"

# ---------------------------------------------------------------- 保存先
WORK_DIR = Path("/content/work")
CHECKPOINT_ROOT = WORK_DIR / "checkpoints"      # checkpoint は <ここ>/<設定名>/<RUN_NAME>/<step>/ に残る
LEROBOT_HOME = WORK_DIR / "lerobot_home"        # 学習時の HF_LEROBOT_HOME
PUSH_TO_HUB = False
HF_MODEL_REPO = "your-hf-name/pi05-drop9-finetuned"   # PUSH_TO_HUB = True のとき
HF_MODEL_PRIVATE = True

# ---------------------------------------------------------------- 版の固定
VLADROP_GIT = "https://github.com/s1ghhh/VLADrop.git"
VLADROP_SHA = "782d3e567c74fa90492588c1ad49351bde61cb7a"
OPENPI_CONFIG = "pi05_libero_plus"            # fork の学習設定の名前
LOCAL_REPO_ID = "local/libero_plus_lerobot"   # その設定が読むデータセットの名前（HF_LEROBOT_HOME の下）
LEROBOT_FOR_V30 = "lerobot==0.4.4"            # fork の LeRobot（v2.1 専用）の代わりに入れる版

print(f"run: {RUN_NAME}")

## 2. ランタイムを確認する

In [ ]:
import json
import os
import shutil
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU ランタイムを選んでください（ランタイム → ランタイムのタイプを変更）")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU {GPU_NAME}  {GPU_GB:.0f} GB")
if GPU_GB < 39:
    raise RuntimeError("全パラメータの学習には VRAM 40 GB 以上（A100）が要ります。ランタイムのタイプで A100 を選んでください")

os.environ.update({
    "HF_HOME": "/content/hf_cache",
    "HF_HUB_DISABLE_TELEMETRY": "1",
})
WORK_DIR.mkdir(parents=True, exist_ok=True)


def run(command, check=True, **kwargs):
    """コマンドを実行し、失敗したら出力の末尾を出して止める。"""
    result = subprocess.run(command, text=True, capture_output=True, **kwargs)
    if check and result.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, command))}\n{(result.stdout + result.stderr)[-4000:]}")
    return result


print(run(["df", "-h", "/content"]).stdout)

## 3. Hugging Face にログインする

token は Colab の Secrets（`HF_TOKEN`）から読む。無ければ入力欄が出る（画面には表示されない）。
token は重みとデータを落とすときとアップロードにだけ使い、学習のプロセスには渡さない。

In [ ]:
from getpass import getpass

from huggingface_hub import login, whoami

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token: ").strip()
login(token=HF_TOKEN, add_to_git_credential=False)
print(f"Hugging Face: {whoami(token=HF_TOKEN)['name']}")

## 4. 初期重みとデータセットを取る

初期重み（約 7.5 GB）は Hugging Face のキャッシュに、データセットは fork の設定が読む場所
（`HF_LEROBOT_HOME/local/libero_plus_lerobot`）に直接落とす。2 回目以降は落とし済みのファイルを飛ばす。

正規化の統計量は計算し直さず、初期重みに同梱のもの（`assets/local/libero_plus_lerobot/norm_stats.json`）をそのまま使う。

In [ ]:
from huggingface_hub import snapshot_download

BASE_MODEL_DIR = Path(snapshot_download(BASE_MODEL_REPO, revision=BASE_MODEL_REVISION, token=HF_TOKEN))
NORM_STATS = BASE_MODEL_DIR / "assets" / LOCAL_REPO_ID / "norm_stats.json"
for path in (BASE_MODEL_DIR / "model.safetensors", BASE_MODEL_DIR / "metadata.pt", NORM_STATS):
    if not path.is_file():
        raise FileNotFoundError(f"初期重みに {path.relative_to(BASE_MODEL_DIR)} がありません")
print(f"base model {BASE_MODEL_REPO}@{BASE_MODEL_REVISION}")
print(f"           {BASE_MODEL_DIR}  ({(BASE_MODEL_DIR / 'model.safetensors').stat().st_size / 1e9:.2f} GB)")

DATASET_DIR = LEROBOT_HOME / LOCAL_REPO_ID
snapshot_download(DATASET_REPO, repo_type="dataset", revision=DATASET_REVISION, local_dir=DATASET_DIR, token=HF_TOKEN)
info = json.loads((DATASET_DIR / "meta/info.json").read_text(encoding="utf-8"))
if info["codebase_version"] != "v3.0":
    raise RuntimeError(f"LeRobot v3.0 のデータセットにしてください（今は {info['codebase_version']}）")
REQUIRED = {
    "observation.images.front": (256, 256, 3),
    "observation.images.wrist": (256, 256, 3),
    "observation.state": (8,),
    "action": (7,),
}
features = info["features"]
problems = [f"{key}: {tuple(features[key]['shape']) if key in features else '無い'} (期待 {shape})"
            for key, shape in REQUIRED.items()
            if key not in features or tuple(features[key]["shape"]) != shape]
if problems:
    raise RuntimeError("データセットの形式が前提と違います:\n  " + "\n  ".join(problems))
if info["fps"] != 20:
    print(f"WARNING: fps={info['fps']}（LIBERO の control_freq は 20）")
print(f"dataset    {DATASET_REPO}@{DATASET_REVISION or 'main'}")
print(f"           {info['total_episodes']:,} episodes / {info['total_frames']:,} frames / "
      f"{info['total_tasks']} tasks / fps {info['fps']}  -> {DATASET_DIR}")
print(f"学習で見るサンプル数 {STEPS * BATCH_SIZE:,}（各フレーム平均 {STEPS * BATCH_SIZE / info['total_frames']:.2f} 回）")

## 5. VLADrop の学習環境を作る

1. VLADrop の fork を取り、固定された依存（Python 3.11・torch 2.7.1・transformers 4.53.2）で fork 専用の環境を作る
   （`uv sync --frozen`）。この notebook の Python とは別の環境になる
2. fork の手順どおり、transformers の一部を fork 同梱のもの（`transformers_replace`）に差し替える
3. v3.0 のデータを読めるようにする。fork の LeRobot を 0.4.4 に入れ替え（torch などの版は動かさない）、
   データを読む関数（`create_torch_dataset`）を次のように書き換える
   - import 先を新しい LeRobot のものにする
   - v3.0 ではタスク一覧が表になったので、番号 → 指示文の辞書に直す
   - 動画は pyav で読む（torchcodec は Colab の FFmpeg を読み込めない）
4. 正規化の統計量を、fork の設定が読む場所（`assets/pi05_libero_plus/local/libero_plus_lerobot/`）へ置く

In [ ]:
VLADROP_DIR = Path("/content/VLADrop")
OPENPI_DIR = VLADROP_DIR / "models/pi0.5/openpi"
GIT_ENV = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")

run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
UV = shutil.which("uv")
if not VLADROP_DIR.is_dir():
    run(["git", "clone", "--quiet", VLADROP_GIT, str(VLADROP_DIR)], env=GIT_ENV)
run(["git", "-C", str(VLADROP_DIR), "checkout", "--quiet", VLADROP_SHA], env=GIT_ENV)

# 1. fork 専用の環境
run([UV, "sync", "--frozen", "--quiet"], cwd=OPENPI_DIR, env=GIT_ENV)
VENV_PY = OPENPI_DIR / ".venv/bin/python"

# 2. transformers の差し替え
transformers_dir = run([str(VENV_PY), "-c", "import os, transformers; print(os.path.dirname(transformers.__file__))"]).stdout.strip()
shutil.copytree(OPENPI_DIR / "src/openpi/models_pytorch/transformers_replace", transformers_dir, dirs_exist_ok=True)

# 3. LeRobot の入れ替え。依存なしで入れ、足りない accelerate と av だけを、fork の版を固定したまま足す
frozen = run([UV, "pip", "freeze", "--python", str(VENV_PY)]).stdout.splitlines()
constraints = WORK_DIR / "vladrop_constraints.txt"
constraints.write_text("".join(f"{line}\n" for line in frozen
                               if line.split("==")[0] in ("torch", "torchvision", "transformers", "numpy", "datasets")))
run([UV, "pip", "install", "--quiet", "--python", str(VENV_PY), "--no-deps", LEROBOT_FOR_V30])
run([UV, "pip", "install", "--quiet", "--python", str(VENV_PY), "-c", str(constraints), "accelerate", "av"])

V30_MARKER = "LeRobot v3.0 patch"
V30_REPLACEMENTS = [
    (
        "import lerobot.common.datasets.lerobot_dataset as lerobot_dataset\n",
        f"try:  # {V30_MARKER}: lerobot >= 0.4 reads LeRobot v3.0 datasets\n"
        "    import lerobot.datasets.lerobot_dataset as lerobot_dataset\n"
        "except ImportError:  # the fork's pinned lerobot (v2.1)\n"
        "    import lerobot.common.datasets.lerobot_dataset as lerobot_dataset\n",
    ),
    (
        "            key: [t / dataset_meta.fps for t in range(action_horizon)] for key in data_config.action_sequence_keys\n"
        "        },\n"
        "    )\n",
        "            key: [t / dataset_meta.fps for t in range(action_horizon)] for key in data_config.action_sequence_keys\n"
        "        },\n"
        f'        video_backend="pyav",  # {V30_MARKER}: torchcodec cannot load the system FFmpeg on Colab\n'
        "    )\n",
    ),
    (
        "        dataset = TransformedDataset(dataset, [_transforms.PromptFromLeRobotTask(dataset_meta.tasks)])\n",
        "        tasks = dataset_meta.tasks\n"
        f"        if not isinstance(tasks, dict):  # {V30_MARKER}: v3.0 keeps tasks as a DataFrame (index = text)\n"
        '            tasks = {int(v): str(k) for k, v in tasks["task_index"].items()}\n'
        "        dataset = TransformedDataset(dataset, [_transforms.PromptFromLeRobotTask(tasks)])\n",
    ),
]
data_loader = OPENPI_DIR / "src/openpi/training/data_loader.py"
source = data_loader.read_text(encoding="utf-8")
if V30_MARKER in source:
    print("data_loader.py は書き換え済み")
else:
    for before, after in V30_REPLACEMENTS:
        if source.count(before) != 1:
            raise RuntimeError(f"data_loader.py に書き換える位置が見つかりません。fork の版を確認してください: {before!r}")
        source = source.replace(before, after, 1)
    data_loader.write_text(source, encoding="utf-8")
    print(f"書き換えた: {data_loader}")

# 4. 正規化の統計量
stats_dst = OPENPI_DIR / "assets" / OPENPI_CONFIG / LOCAL_REPO_ID / "norm_stats.json"
stats_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(NORM_STATS, stats_dst)

print(run([str(VENV_PY), "-c",
           "import sys, importlib.metadata as m; print('Python', sys.version.split()[0]); "
           "[print(f'{p:<12} {m.version(p)}') for p in ('torch', 'transformers', 'lerobot', 'accelerate', 'av')]"]).stdout)

## 6. 追加学習する

fork の `train_pytorch.py` を別プロセスで動かし、`LOG_FREQ` ごとに loss と learning rate を表示する。
全部の出力は `WORK_DIR/train-<名前>.log` に残る。

- 初期重みを読み込み、`LLM_DROP_LIST` の層を落とした構造のまま学習する（落とした層は使われない）
- 学習のプロセスはオフライン（`HF_HUB_OFFLINE=1`）で動かし、HF の token は渡さない
- まず `SMOKE_STEPS` だけ回して最後まで通る（メモリに載る）ことを確かめ、その checkpoint は消す。次に本番を回す
- 最初の 1 回は動画の読み込みの準備で数分かかる

In [ ]:
import re
from collections import deque

TRAIN_ENV = {key: value for key, value in os.environ.items() if key != "HF_TOKEN"}
TRAIN_ENV.update(HF_LEROBOT_HOME=str(LEROBOT_HOME), HF_HUB_OFFLINE="1", TOKENIZERS_PARALLELISM="false",
                 WANDB_MODE="disabled", XLA_PYTHON_CLIENT_PREALLOCATE="false", PYTHONUNBUFFERED="1")


def train(exp_name, steps, warmup_steps):
    """fork の train_pytorch.py で学習し、checkpoint のディレクトリを返す。"""
    command = [
        str(VENV_PY), "-m", "torch.distributed.run", "--standalone", "--nnodes=1", "--nproc_per_node=1",
        "scripts/train_pytorch.py", OPENPI_CONFIG,
        "--exp-name", exp_name,
        "--pytorch-weight-path", str(BASE_MODEL_DIR),
        "--model.llm-drop-attn-list", *map(str, LLM_DROP_LIST),
        "--model.llm-drop-mlp-list", *map(str, LLM_DROP_LIST),
        "--batch-size", str(BATCH_SIZE),
        "--num-train-steps", str(steps),
        "--save-interval", str(steps),
        "--log-interval", "1",
        "--num-workers", str(NUM_WORKERS),
        "--lr-schedule.warmup-steps", str(warmup_steps),
        "--lr-schedule.peak-lr", str(PEAK_LR),
        "--lr-schedule.decay-steps", str(steps),
        "--lr-schedule.decay-lr", str(DECAY_LR),
        "--ema-decay", str(EMA_DECAY),
        "--checkpoint-base-dir", str(CHECKPOINT_ROOT),
        "--no-wandb-enabled",
        "--overwrite",
        "--gradient-checkpointing" if GRADIENT_CHECKPOINTING else "--no-gradient-checkpointing",
    ]
    log_path = WORK_DIR / f"train-{exp_name}.log"
    print(f"[{exp_name}] {steps} steps / batch {BATCH_SIZE} / lr {PEAK_LR:g} -> {DECAY_LR:g}  (log {log_path})")
    recent = deque(maxlen=60)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=OPENPI_DIR, env=TRAIN_ENV, text=True, bufsize=1,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in process.stdout:
            log.write(line)
            recent.append(line.rstrip())
            match = re.search(r"step=(\d+) loss=([0-9.]+) lr=([0-9.e+-]+)", line)
            if match:
                step = int(match.group(1))
                if step % LOG_FREQ == 0 or step >= steps - 1:
                    print(f"  step {step:>5}  loss {match.group(2)}  lr {match.group(3)}", flush=True)
            elif "Loaded PyTorch weights" in line or "Strict loading failed" in line:
                print("  " + line.strip(), flush=True)
    if process.wait() != 0:
        print("\n".join(recent))
        oom = "CUDA out of memory" in log_path.read_text(encoding="utf-8", errors="replace")
        raise RuntimeError(f"{exp_name} の学習が失敗しました（exit {process.returncode}"
                           f"{'、GPU のメモリ不足' if oom else ''}）")
    return CHECKPOINT_ROOT / OPENPI_CONFIG / exp_name


if SMOKE_STEPS:
    train("smoke", SMOKE_STEPS, max(1, SMOKE_STEPS // 5))
    shutil.rmtree(CHECKPOINT_ROOT / OPENPI_CONFIG / "smoke", ignore_errors=True)
    print("smoke: 最後まで通りました")

In [ ]:
run_dir = train(RUN_NAME, STEPS, WARMUP_STEPS)
FINAL_CHECKPOINT = max((p for p in run_dir.iterdir() if p.name.isdigit()), key=lambda p: int(p.name))
for path in sorted(p for p in FINAL_CHECKPOINT.rglob("*") if p.is_file()):
    print(f"  {path.relative_to(FINAL_CHECKPOINT)}  {path.stat().st_size / 1e9:.2f} GB")
print(f"checkpoint {FINAL_CHECKPOINT}")

## 7. Hugging Face へアップロードする（任意）

`PUSH_TO_HUB = True` のときだけ動く。既定では private の model repo に、`<RUN_NAME>/<step>/` として置く。
推論に要る `model.safetensors`・`metadata.pt`・`assets/` だけを上げ、`optimizer.pt` は上げない。

In [ ]:
from huggingface_hub import HfApi

if PUSH_TO_HUB:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_MODEL_REPO, repo_type="model", private=HF_MODEL_PRIVATE, exist_ok=True)
    path_in_repo = f"{RUN_NAME}/{FINAL_CHECKPOINT.name}"
    commit = api.upload_folder(
        folder_path=str(FINAL_CHECKPOINT), repo_id=HF_MODEL_REPO, repo_type="model",
        path_in_repo=path_in_repo, ignore_patterns=["optimizer.pt"],
        commit_message=(f"VLADrop pi0.5 Drop9 full fine-tune: {DATASET_REPO}, {STEPS} steps, batch {BATCH_SIZE}, "
                        f"lr {PEAK_LR:g} -> {DECAY_LR:g} (VLADrop {VLADROP_SHA[:8]})"),
    )
    print(f"https://huggingface.co/{HF_MODEL_REPO}/tree/main/{path_in_repo}  revision {commit.oid}")
else:
    print("PUSH_TO_HUB = False なのでアップロードしません")

## 学習済みモデルについて

checkpoint は openpi（PyTorch）の形式で、[SmolVLA 版](smolvla_libero_finetune.ipynb) や
[π0.5 版](pi05_libero_finetune.ipynb) のように LeRobot の `from_pretrained` では読めない。
推論では、学習と同じ `LLM_DROP_LIST` を fork の推論側にも明示して渡す。

parc2026 では fork の推論サーバでこの checkpoint を LIBERO-Plus で動かして評価し、動画を撮った
（上の表の 16 問）。そのままでは層を落とした構造の推論が通らず、推論側に手当てが要ったため、
評価の手順はこの notebook には入れていない。